In [1]:
# =========================================================
# FINAL COMPETITION PIPELINE (STABLE + HIGH ROI)
# Expected runtime: ~7–9 hours (1x T4)
# =========================================================

import os, random, glob, time, copy
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets, models

from sklearn.model_selection import StratifiedKFold
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

# ================= CONFIG =================
SEED = 42
BATCH_SIZE = 32
N_FOLDS = 3
BASE_LR = 1e-4
BACKBONE_LR = 1e-5
EMA_DECAY = 0.999
TIME_LIMIT = 11 * 3600
START_TIME = time.time()

SCENE_ROOT = "/kaggle/input/competitions/cse-281-spring-26-scene-style-classification/StyleClassificationIndoors/StyleClassificationIndoors"
TRAIN_DIR = os.path.join(SCENE_ROOT, "train")
TEST_DIR = os.path.join(SCENE_ROOT, "test")

STAGES = [
    {"epochs": 12, "size": 224},
    {"epochs": 10, "size": 288}
]

# ================= SEED =================
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
device = torch.device("cuda:0")

# ================= AUG =================
def get_transforms(sz, train=True):
    mean, std = [0.485,0.456,0.406],[0.229,0.224,0.225]
    
    if train:
        return transforms.Compose([
            transforms.RandomResizedCrop(sz, scale=(0.6,1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandAugment(2,7),
            transforms.ToTensor(),
            transforms.Normalize(mean,std),
        ])
    else:
        return transforms.Compose([
            transforms.Resize(int(sz*1.1)),
            transforms.CenterCrop(sz),
            transforms.ToTensor(),
            transforms.Normalize(mean,std),
        ])

# ================= MIXUP =================
def mixup(x,y,alpha=0.2):
    if np.random.rand() > 0.5:
        return x,y,y,1.0
    lam = np.random.beta(alpha,alpha)
    idx = torch.randperm(x.size(0)).to(device)
    return lam*x+(1-lam)*x[idx], y, y[idx], lam

# ================= MODEL =================
class Model(nn.Module):
    def __init__(self,n):
        super().__init__()
        self.backbone = models.convnext_base(weights='IMAGENET1K_V1')
        f = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(f,n)
        )
    def forward(self,x): return self.backbone(x)

# ================= EMA =================
class EMA:
    def __init__(self,model,decay):
        self.model=model
        self.decay=decay
        self.shadow={n:p.clone().detach() for n,p in model.named_parameters() if p.requires_grad}
        self.backup={}
        
    def update(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad:
                self.shadow[n]=self.decay*self.shadow[n]+(1-self.decay)*p.data
                
    def apply(self):
        self.backup={}
        for n,p in self.model.named_parameters():
            if p.requires_grad:
                self.backup[n]=p.data.clone()
                p.data.copy_(self.shadow[n])
                
    def restore(self):
        for n,p in self.model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])

# ================= DATA =================
full_ds = datasets.ImageFolder(TRAIN_DIR)
num_classes = len(full_ds.classes)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
test_paths = sorted(glob.glob(os.path.join(TEST_DIR,"*.*")))

all_probs = []

# ================= TRAIN =================
for fold,(t_idx,v_idx) in enumerate(skf.split(np.zeros(len(full_ds)), full_ds.targets)):
    
    if time.time()-START_TIME > TIME_LIMIT: break
    print(f"\n--- FOLD {fold+1} ---")
    
    model = Model(num_classes).to(device)
    ema = EMA(model, EMA_DECAY)
    
    # differential LR
    params = [
        {'params':[p for n,p in model.named_parameters() if "classifier" not in n],'lr':BACKBONE_LR},
        {'params':[p for n,p in model.named_parameters() if "classifier" in n],'lr':BASE_LR}
    ]
    
    optimizer = torch.optim.AdamW(params, weight_decay=0.05)
    scaler = GradScaler()
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    best_acc = 0
    best_weights = None
    
    for stage in STAGES:
        
        train_loader = DataLoader(
            Subset(datasets.ImageFolder(TRAIN_DIR, get_transforms(stage["size"],True)), t_idx),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
        )
        
        val_loader = DataLoader(
            Subset(datasets.ImageFolder(TRAIN_DIR, get_transforms(stage["size"],False)), v_idx),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=2
        )
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=stage["epochs"]
        )
        
        for epoch in range(stage["epochs"]):
            model.train()
            
            for x,y in tqdm(train_loader, leave=False):
                x,y = x.to(device), y.to(device)
                
                x,y1,y2,lam = mixup(x,y)
                
                optimizer.zero_grad(set_to_none=True)
                
                with autocast():
                    out = model(x)
                    loss = lam*criterion(out,y1)+(1-lam)*criterion(out,y2)
                
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                
                ema.update()
            
            # VALIDATE EMA
            ema.apply()
            model.eval()
            
            correct,total = 0,0
            with torch.no_grad():
                for x,y in val_loader:
                    x,y = x.to(device), y.to(device)
                    with autocast():
                        out = model(x)
                    correct += (out.argmax(1)==y).sum().item()
                    total += y.size(0)
            
            acc = correct/total
            
            if acc > best_acc:
                best_acc = acc
                best_weights = copy.deepcopy(ema.shadow)
                print(f"⭐ {best_acc:.4f}")
            
            ema.restore()
            scheduler.step()
    
    # LOAD BEST
    for n,p in model.named_parameters():
        if p.requires_grad:
            p.data.copy_(best_weights[n])
    
    # ================= INFERENCE =================
    model.eval()
    fold_probs = []
    
    t1 = get_transforms(288, False)
    t2 = transforms.Compose([
        transforms.Resize(320),
        transforms.CenterCrop(288),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ])
    
    with torch.no_grad():
        for pth in tqdm(test_paths, leave=False):
            try:
                img = Image.open(pth).convert("RGB")
                
                x1 = t1(img).unsqueeze(0).to(device)
                x2 = torch.flip(x1,[3])
                x3 = t2(img).unsqueeze(0).to(device)
                
                with autocast():
                    p1 = torch.softmax(model(x1),1)
                    p2 = torch.softmax(model(x2),1)
                    p3 = torch.softmax(model(x3),1)
                
                pred = (p1+p2+p3)/3
                fold_probs.append(pred.cpu().numpy())
            except:
                fold_probs.append(np.zeros((1,num_classes)))
    
    all_probs.append(np.vstack(fold_probs))
    
    del model
    torch.cuda.empty_cache()

# ================= SUBMIT =================
final_probs = np.mean(all_probs, axis=0)
labels = final_probs.argmax(1)

pd.DataFrame({
    "ImageName":[os.path.basename(p) for p in test_paths],
    "label":labels
}).to_csv("submission.csv", index=False)

print("✅ DONE")


--- FOLD 1 ---
Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth


100%|██████████| 338M/338M [00:01<00:00, 228MB/s]
/tmp/ipykernel_23/3208350769.py:140: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
  0%|          | 0/275 [00:00<?, ?it/s]/tmp/ipykernel_23/3208350769.py:172: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_23/3208350769.py:190: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


⭐ 0.0786


⭐ 0.2103


⭐ 0.3118


...

⭐ 0.4500


⭐ 0.4550


⭐ 0.4589


⭐ 0.4630


✅ DONE
